# HW 4 — Regex with AI: Generate It, Then Verify It

## Overview
This notebook uses an AI-generated regular expression to extract subsidiary names and locations from Federal Signal Corporation's FY2024 SEC Exhibit 21. The result is then verified against manually established ground truth and several edge cases.

## Part A — Generate It

**Goal:** download the SEC Exhibit 21, inspect the raw HTML, document the AI prompt/code, and run the AI-generated regex.

### A1. Student and Filing Information

In [ ]:
# --- Cell 1: you and your claimed filing ---

name        = "리오디노 라이한"
student_id  = "50261893"
company     = "Federal Signal Corporation"
fiscal_year = "FY2024"
claimed_url = "https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm"

print("Name:", name)
print("Student ID:", student_id)
print("Company:", company)
print("Fiscal Year:", fiscal_year)
print("Exhibit 21 URL:", claimed_url)

### A2. Download SEC Exhibit 21

In [ ]:
# --- Cell 2: fetch the exhibit ---

import requests

URL = "https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm"

headers = {
    "User-Agent": "Riodino Raihan riodinoraihan@gmail.com"
}

response = requests.get(URL, headers=headers)

print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))

html = response.text

### A3. Inspect the Downloaded HTML

The first part of the downloaded HTML is displayed below to confirm that the SEC Exhibit 21 was successfully retrieved.

In [ ]:
print(html[:500])

### A4. AI Tool, Exact Prompt, and Exact AI-Generated Code

**AI Tool / Model:** [Enter the exact Gemini model you used]

**Exact Prompt:**

[PASTE the exact prompt you actually sent to Gemini]

**Exact AI-Generated Code:**

[PASTE the exact code Gemini actually generated]

> **Important:** Do not rewrite the prompt or code in this documentation section. This section should show what was actually generated by the AI.

### A5. Run the AI-Generated Regex

In [ ]:
import re
import html as html_lib

# (Assuming the `html` variable is already defined and contains the raw HTML string)
# html = "..."

def extract_subsidiaries(html_content):
    # Regex patterns for finding rows, cells, and removing HTML tags
    # We account for possible whitespace in closing tags (e.g., </tr >)
    row_regex = re.compile(r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>', re.IGNORECASE | re.DOTALL)
    cell_regex = re.compile(r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>', re.IGNORECASE | re.DOTALL)
    tag_regex = re.compile(r'<[^>]+>')
    space_regex = re.compile(r'\s+')
    
    extracted_pairs = []
    
    # 1. Extract all <tr> blocks
    rows = row_regex.findall(html_content)
    
    for row in rows:
        # 2. Extract all <td> blocks within the current row
        cells = cell_regex.findall(row)
        
        cleaned_cells = []
        for cell in cells:
            # Replace HTML tags with a space to prevent words from fusing together
            text = tag_regex.sub(' ', cell)
            
            # Unescape HTML entities (converts &amp; to &, &#160; to space, etc.)
            text = html_lib.unescape(text)
            
            # Replace multiple whitespace characters (newlines, tabs, etc.) with a single space
            text = space_regex.sub(' ', text).strip()
            
            # If the cell contains text after cleaning, add it to our valid cells list
            if text:
                cleaned_cells.append(text)
        
        # 3. Process rows with at least 2 non-empty cells
        if len(cleaned_cells) >= 2:
            # First relevant cell is the Subsidiary Name, the last is the Location
            subsidiary_name = cleaned_cells[0]
            location = cleaned_cells[-1]
            
            # Ignore table header rows that usually contain words like "Subsidiary", "Name", "Jurisdiction"
            name_lower = subsidiary_name.lower()
            loc_lower = location.lower()
            if ("subsidiary" in name_lower or "name of" in name_lower or 
                "jurisdiction" in loc_lower or "state" in loc_lower):
                continue
                
            extracted_pairs.append((subsidiary_name, location))
            
    return extracted_pairs

# --- Execution ---
pairs = extract_subsidiaries(html)

# --- Output Results ---
print(f"Total extracted pairs: {len(pairs)}\n")

print("First 10 pairs:")
for i, (name, loc) in enumerate(pairs[:10], start=1):
    print(f"{i}. {name} | {loc}")

## Part B — Verify It

**Goal:** establish ground truth and check whether the AI-generated extraction is correct.

### B1. Ground Truth

I manually counted the named subsidiary rows in the Exhibit 21 table. The manual count is **33 subsidiaries**, excluding the table header and the footnote below the table.

**Manual count = 33**

**AI-generated regex count = 33**

Therefore, the counts match.

### B2. Count Verification

In [ ]:
print("Manual count:", 33)
print("AI-generated regex count:", len(pairs))
print("Counts match:", len(pairs) == 33)

### B3. Complete Extraction Result

In [ ]:
print("All extracted pairs:")
for i, pair in enumerate(pairs, start=1):
    print(f"{i}. {pair[0]} | {pair[1]}")

### B4. First and Last Row Verification

In [ ]:
print("First extracted row:")
print(pairs[0])

print("\nLast extracted row:")
print(pairs[-1])

### B5. Awkward Row Tests

These tests check subsidiary names containing characters that can sometimes cause extraction problems, including `&` and parentheses.

**Test 1 — Ampersand (`&`)**

In [ ]:
for pair in pairs:
    if "OSW Equipment" in pair[0]:
        print(pair)

**Test 2 — Ampersand (`&`) in a longer name**

In [ ]:
for pair in pairs:
    if "Truck Bodies & Equipment" in pair[0]:
        print(pair)

**Test 3 — Parentheses (`(...)`)**

In [ ]:
for pair in pairs:
    if "Victor Industrial Equipment" in pair[0]:
        print(pair)

### B6. Missing-Location Edge Case

The original filing does not contain a named subsidiary row with a missing location. Therefore, a small artificial HTML example is used only to test how the function behaves when the location cell is empty.

In [ ]:
test_html = """
<table>
<tr>
<td>Test Subsidiary</td>
<td></td>
</tr>
</table>
"""

test_pairs = extract_subsidiaries(test_html)

print("Missing-location test:")
print(test_pairs)

### B7. Path 2 — Proving It

I established the ground truth manually by counting the subsidiary rows in Federal Signal Corporation's FY2024 Exhibit 21. The filing contains 33 named subsidiary rows, and the AI-generated regex also extracted 33 pairs, so the counts match.

The first extracted pair was `Crysteel Manufacturing, Inc. — Minnesota`, which matches the first subsidiary row in the filing. The last extracted pair was `Work Equipment Ltd. — Canada`, which also matches the last subsidiary row.

I deliberately tested several awkward cases. `OSW Equipment & Repair, LLC — Washington` and `Truck Bodies & Equipment International, Inc. — Delaware` confirmed that the regex handled subsidiary names containing an ampersand (`&`). `Victor Industrial Equipment (PTY) Limited — South Africa` confirmed that parentheses in a subsidiary name were also handled correctly.

The original filing does not contain a named subsidiary row with a missing location, so I did not invent one. Instead, I created a small deliberate edge-case test containing a subsidiary with an empty location cell. The result was an empty list (`[]`), showing that the code does not return an incomplete subsidiary-location pair as valid data.

## Part C — Fixing and Explaining It

**Goal:** determine whether a fix is necessary and explain how the regular expressions work.

### C1. Fix

No fix was needed. The original AI-generated regex successfully extracted all 33 subsidiary-location pairs from the Federal Signal Corporation FY2024 Exhibit 21. The first and last rows were correctly extracted, and the deliberately tested awkward cases were also handled correctly.

Because the verification results matched the manually established ground truth, I did not modify the AI-generated regex.

### C2. Regex Explanation

#### C2.1 Row Pattern

The row pattern is used to find each HTML table row (`<tr>...</tr>`).

In [ ]:
row_regex = re.compile(
    r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>',
    re.IGNORECASE | re.DOTALL
)

**Explanation:**

- `<\s*tr` identifies an opening `<tr>` tag while allowing optional whitespace.
- `[^>]*` allows attributes inside the opening tag.
- `(.*?)` captures the contents of the row without taking more text than necessary.
- `<\s*/\s*tr\s*>` matches the closing `</tr>` tag.
- `re.IGNORECASE` allows different capitalization of HTML tags.
- `re.DOTALL` allows the match to include line breaks.

#### C2.2 Cell Pattern

The cell pattern is used to find individual table cells (`<td>...</td>`) inside each row.

In [ ]:
cell_regex = re.compile(
    r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>',
    re.IGNORECASE | re.DOTALL
)

**Explanation:**

- `<\s*td` identifies an opening `<td>` tag.
- `[^>]*` allows attributes inside the opening tag.
- `(.*?)` captures the cell contents.
- `<\s*/\s*td\s*>` matches the closing `</td>` tag.
- `re.IGNORECASE` and `re.DOTALL` provide the same flexibility described for the row pattern.

#### C2.3 HTML Tag Removal

In [ ]:
tag_regex = re.compile(r'<[^>]+>')

**Explanation:**

- `<` marks the beginning of an HTML tag.
- `[^>]+` matches one or more characters that are not `>`.
- `>` marks the end of the tag.

This removes HTML tags while keeping the text contained inside them.

#### C2.4 Whitespace Normalization

In [ ]:
space_regex = re.compile(r'\s+')

**Explanation:**

`\s+` matches one or more whitespace characters, including spaces, tabs, and line breaks. Replacing them with a single space makes the extracted names and locations cleaner and more consistent.

#### C2.5 Selecting the Name and Location

After extracting the `<td>` cells, the code cleans every cell and keeps only non-empty cells. The first non-empty cell is assigned as the subsidiary name, while the last non-empty cell is assigned as the location.

This is important for this filing because the table contains empty cells used for spacing. Therefore, the code does not simply assume that the second physical `<td>` is always the location.

### C3. Reflection

The AI-generated regex worked correctly on the Federal Signal Corporation FY2024 Exhibit 21, so I did not find a failure that needed to be fixed. The verification process was important because I could confirm the result by comparing the extracted count with my manual ground truth and by checking the first, last, and awkward rows. Even if I could not understand the regex syntax, I could still detect a problem by comparing the AI output with the original filing and checking whether any subsidiaries were missing or incorrectly extracted. I would also test the same approach on a differently structured Exhibit 21 because a regex that works for one HTML structure may not work for every SEC filing.